# Beta 分布的矩估计（GMM）

> 对应代码：`MathStatsCode/code_in_notes/Chap.10/MM_beta.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：用一阶矩与二阶矩条件对 Beta(1.2,0.5) 的参数做矩估计，并模拟1000次观察估计量的分布。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.10`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.10")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

本 notebook 用**矩估计（GMM）**估计 Beta 分布的参数，并模拟估计量的抽样分布。

**第①步：定义估计程序。** `program est_beta, eclass` 定义一个返回估计结果（eclass）的程序，`syntax varlist(max=1) [, alpha(real 1) beta(real 1)]` 接受一个变量与两个初始值参数。内部用 `gmm` 命令设定两个矩条件——Beta(α,β) 的理论一阶矩为 α/(α+β)、二阶矩为 αβ(α+β+1)/(α+β)²(α+β+1)：即程序中两条 `(样本矩-理论矩)` 形式的矩条件；`winit(identity)` 表示先用单位矩阵作为权重矩阵。两个矩条件、两个参数，属于恰好识别，可用单位权重矩阵直接求解。

**第②步：把 GMM 结果“接回”eclass 体系。** GMM 不是标准估计命令，这里用 `mat b=e(b)`/`mat V=e(V)` 取出系数与协方差矩阵，再用 `ereturn post` 写入标准 e(b)/e(V)，使后续 `_b[/alpha]` 等访问方式可用。

**第③步：模拟。** `frame create simulation` 新建结果框，用 `forvalues` 循环1000次：每次生成100个 Beta(1.2,0.5) 观测并用 `est_beta` 估计，`frame post` 把两个估计值写入结果框；最后 `su` 查看估计量的均值与标准差（均值应接近真值 1.2 与 0.5，标准差反映估计精度）。


In [ ]:
%%stata
clear all
set more off
set seed 20221130
cap program drop est_beta
program est_beta, eclass
	syntax varlist(max=1) [, alpha(real 1) beta(real 1)]
	tempvar x2
	quietly{
		gen `x2'=`varlist'^2
		gmm (`varlist'-({alpha=`alpha'}/({alpha}+{beta=`beta'}))) (`x2'-({alpha}*{beta}+{alpha}^2*({alpha}+{beta}+1))/(({alpha}+{beta})^2*({alpha}+{beta}+1))), winit(identity)
	}
	mat b=e(b)
	mat V=e(V)
	ereturn post b V
	ereturn display
end

frame create simulation alpha beta
forvalues i=1/1000{
	clear
	quietly{
		set obs 100
		gen x=rbeta(1.2,0.5)
		est_beta x
		frame post simulation (_b[/alpha]) (_b[/beta])
	}
}
frame simulation: su
